⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Migration: weather.txt\n",
    "**Source Task:** SCEN_TASK_NO 1\n",
    "**Target:** OdiInvokeRESTfulService conversion to Databricks Python/SQL\n",
    "**Timestamp:** 2024-05-22"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\")\n",
    "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
    "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
    "dbutils.widgets.text(\"ODI_SESS_NO\", \"\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### ETL Parameters"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "CREATE OR REPLACE TEMPORARY VIEW v_etl_session_params AS\n",
    "SELECT \n",
    "    '${ETL_JOB_TYPE}' AS etl_job_type,\n",
    "    '${ODI_SESS_NO}' AS odi_sess_no,\n",
    "    CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id;\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "display(spark.sql(\"SELECT * FROM v_etl_session_params\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### REST API Invocation (OdiInvokeRESTfulService)\n",
    "Converted from LS_Weather logical schema to Python requests logic."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import requests\n",
    "import json\n",
    "import os\n",
    "\n",
    "# Endpoint details derived from LS_Weather and -CONTEXT=DEV\n",
    "# In a real migration, endpoints are stored in Databricks Secrets or Environment Variables\n",
    "api_url = \"https://api.weather.example.com/v1/forecast\" \n",
    "response_path = \"/tmp/weather.json\"\n",
    "\n",
    "try:\n",
    "    # Invoke RESTful Service\n",
    "    response = requests.get(api_url, timeout=60)\n",
    "    response.raise_for_status()\n",
    "    \n",
    "    # Save response to local filesystem (mimicking C:\\\ODI\\\weather.json)\n",
    "    data = response.json()\n",
    "    os.makedirs(os.path.dirname(response_path), exist_ok=True)\n",
    "    with open(response_path, 'w') as f:\n",
    "        json.dump(data, f)\n",
    "        \n",
    "    print(f\"Successfully saved REST response to {response_path}\")\n",
    "except Exception as e:\n",
    "    print(f\"Error invoking REST service: {str(e)}\")\n",
    "    raise e"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Staging Load\n",
    "Optional: Load the JSON data into a Delta staging table."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "DROP TABLE IF EXISTS workspace.ls_weather.c_weather_stg;\n",
    "\n",
    "CREATE TABLE workspace.ls_weather.c_weather_stg \n",
    "USING JSON \n",
    "LOCATION '/tmp/weather.json';\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Validation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "SELECT COUNT(*) FROM workspace.ls_weather.c_weather_stg;\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Cleanup"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "-- MAGIC %sql\n",
    "DROP TABLE IF EXISTS workspace.ls_weather.c_weather_stg;\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "**Conversion Notes:**\n",
    "- `OdiInvokeRESTfulService` was converted to a Python `requests` call.\n",
    "- Logical Schema `LS_Weather` is represented by the `workspace.ls_weather` schema namespace.\n",
    "- File output `C:\\\ODI\\\weather.json` was mapped to `/tmp/weather.json`.\n",
    "- Semicolons and MAGIC SQL tags applied per requirements."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}